# Day 66: MLOps Basics & Model Deployment
From Notebook to Production

**Day 66 of 369-day Python & AI Learning Path**

## Welcome to MLOps!

Congratulations on making it to Day 66! By now, you've mastered the art of building powerful machine learning models - from data preprocessing and feature engineering to training sophisticated algorithms and evaluating their performance. But here's a crucial truth that separates hobbyist data scientists from professional AI engineers: **building a great model is only half the battle.**

The real magic happens when your model leaves the comfort of your Jupyter notebook and starts creating value in the real world. Whether it's predicting customer churn for a business, diagnosing medical images in a hospital, or recommending products to millions of users, your model needs to be deployed, served, monitored, and maintained. This is where **MLOps** (Machine Learning Operations) comes into play.

Today, we enter the exciting phase of production ML engineering. We'll learn how to take models from experimental notebooks into robust production systems. You'll discover the essential practices of model serialization, create reproducible training pipelines, explore deployment strategies, and understand the foundational principles that keep ML systems reliable and scalable. This is the critical bridge between "it works on my machine" and "it's powering a live service."

By the end of this notebook, you'll have a solid understanding of how to package your models for deployment, serve predictions through simple APIs, version your artifacts, and implement basic monitoring. These skills are non-negotiable for anyone serious about applied AI and are exactly what hiring managers look for in ML Engineer and MLOps roles. Let's transform your models from experiments into production-ready assets! 


## Table of Contents

1. [What is MLOps and Why It Matters](#1-what-is-mlops-and-why-it-matters)
2. [Model Serialization & Saving](#2-model-serialization--saving)
3. [Creating a Reproducible Training Script](#3-creating-a-reproducible-training-script)
4. [Building a Simple Prediction Pipeline](#4-building-a-simple-prediction-pipeline)
5. [Basic Model Deployment Options](#5-basic-model-deployment-options)
6. [Versioning Models and Data](#6-versioning-models-and-data)
7. [Monitoring and Logging Basics](#7-monitoring-and-logging-basics)
8. [End-to-End Example: Train -> Save -> Load -> Predict](#8-end-to-end-example)
9. [Best Practices for Moving from Notebook to Production](#9-best-practices)
10. [Hands-On Exercises](#hands-on-exercises)
11. [Solutions & Best Practices](#solutions--best-practices)


## 1. What is MLOps and Why It Matters

**MLOps** is the practice of applying DevOps principles to machine learning systems. It encompasses the entire lifecycle: data preparation, model training, deployment, monitoring, and retraining.

### The ML Lifecycle Gap
Traditional software has code + data -> application. ML adds a third dimension: **the model**, which is a function of code, data, AND training process. This makes deployment significantly more complex.

### Key MLOps Challenges:
- **Reproducibility**: Can you recreate this exact model six months later?
- **Scalability**: Can your inference handle 10x traffic spikes?
- **Monitoring**: Is your model's performance degrading in production?
- **Versioning**: Which data version trained which model version?
- **Governance**: Who changed what, when, and why?

### Why This Matters:
- 87% of ML projects never make it to production (Gartner)
- Models in production degrade over time (data/concept drift)
- Debugging production ML is harder than traditional software
- Regulatory requirements demand traceability and explainability

**Today we build the foundation that solves these problems.**`n

## 2. Model Serialization & Saving

Model serialization is the process of converting a trained model object into a format that can be saved to disk and loaded later. This is the first step toward deployment.

### Common Serialization Formats:
- **Pickle**: Python's built-in serialization (flexible but Python-only)
- **Joblib**: Optimized for large numpy arrays (scikit-learn standard)
- **ONNX**: Open standard for cross-platform deployment
- **JSON**: For model coefficients/weights in some frameworks
- **SavedModel/ HDF5**: TensorFlow/Keras native formats

We'll focus on **joblib** (scikit-learn standard) and introduce **ONNX** as a preview for cross-platform deployment.


In [ ]:
# Standard imports for model serialization
import joblib
import pickle
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print("Joblib version:", joblib.__version__)

In [ ]:
# Create a sample dataset and train a model for serialization demo
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=3,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print(f"Train Accuracy: {train_score:.4f}")
print(f"Test Accuracy: {test_score:.4f}")
print(f"Model type: {type(model).__name__}")

In [ ]:
# Method 1: Save with joblib (recommended for scikit-learn)
import os

os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/random_forest_v1.joblib')
print("Model saved to: models/random_forest_v1.joblib")

with open('models/random_forest_v1.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved to: models/random_forest_v1.pkl")

joblib_size = os.path.getsize('models/random_forest_v1.joblib')
pickle_size = os.path.getsize('models/random_forest_v1.pkl')
print(f"Joblib size: {joblib_size / 1024:.2f} KB")
print(f"Pickle size: {pickle_size / 1024:.2f} KB")

In [ ]:
# Load models back
loaded_model_joblib = joblib.load('models/random_forest_v1.joblib')
print("Model loaded via joblib")

with open('models/random_forest_v1.pkl', 'rb') as f:
    loaded_model_pickle = pickle.load(f)
print("Model loaded via pickle")

sample = X_test[:5]
pred_joblib = loaded_model_joblib.predict(sample)
pred_pickle = loaded_model_pickle.predict(sample)
pred_original = model.predict(sample)

print(f"\nPredictions match (joblib vs original): {np.array_equal(pred_joblib, pred_original)}")
print(f"Predictions match (pickle vs original): {np.array_equal(pred_pickle, pred_original)}")
print(f"Sample predictions: {pred_joblib}")

In [ ]:
# Save the complete pipeline (model + preprocessing)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_scaled = RandomForestClassifier(n_estimators=100, random_state=42)
model_scaled.fit(X_train_scaled, y_train)

pipeline_artifacts = {
    'model': model_scaled,
    'scaler': scaler,
    'feature_names': [f'feature_{i}' for i in range(X.shape[1])],
    'training_metadata': {
        'n_samples': len(X_train),
        'n_features': X.shape[1],
        'train_score': model_scaled.score(X_train_scaled, y_train),
        'test_score': model_scaled.score(X_test_scaled, y_test),
        'model_version': '1.0.0',
        'training_date': pd.Timestamp.now().isoformat(),
    },
}

joblib.dump(pipeline_artifacts, 'models/complete_pipeline_v1.joblib')
print("Complete pipeline saved (model + scaler + metadata)")
print("\nSaved artifacts:")
for key in pipeline_artifacts.keys():
    print(f" - {key}")

## 3. Creating a Reproducible Training Script

A reproducible training script is the backbone of MLOps. It should be deterministic, configurable, and self-documenting. No more "magic cells" scattered across notebooks!

### Key Principles:
- **Fixed random seeds** everywhere
- **Configuration objects** instead of hardcoded values
- **Logging** of all parameters and metrics
- **Artifact saving** with timestamps and versions
- **Environment specification** (requirements.txt, environment.yml)


In [ ]:
# Production-style training script structure
import logging
from dataclasses import dataclass
from typing import Dict, Any, Tuple
import json

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

@dataclass
class ModelConfig:
    n_estimators: int = 100
    max_depth: int = 10
    random_state: int = 42
    test_size: float = 0.2
    model_name: str = "random_forest_classifier"
    version: str = "1.0.0"

def set_seeds(seed: int = 42) -> None:
    np.random.seed(seed)
    import random
    random.seed(seed)
    logger.info(f"Random seeds set to {seed}")

def load_and_split_data(config: ModelConfig) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    X, y = make_classification(
        n_samples=1000, n_features=10, n_informative=5,
        n_redundant=3, random_state=config.random_state
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config.test_size, random_state=config.random_state
    )
    logger.info(f"Data loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")
    return X_train, X_test, y_train, y_test

def train_model(X_train: np.ndarray, y_train: np.ndarray, config: ModelConfig):
    model = RandomForestClassifier(
        n_estimators=config.n_estimators,
        max_depth=config.max_depth,
        random_state=config.random_state
    )
    model.fit(X_train, y_train)
    logger.info(f"Model trained: {config.model_name}")
    return model

def evaluate_model(model, X_train, X_test, y_train, y_test) -> Dict[str, float]:
    metrics = {
        'train_accuracy': float(model.score(X_train, y_train)),
        'test_accuracy': float(model.score(X_test, y_test)),
    }
    logger.info(f"Metrics: {metrics}")
    return metrics

config = ModelConfig()
set_seeds(config.random_state)
X_train, X_test, y_train, y_test = load_and_split_data(config)
model = train_model(X_train, y_train, config)
metrics = evaluate_model(model, X_train, X_test, y_train, y_test)

print("\nReproducible training completed")
print(f"Config: {config}")

In [ ]:
# Save training artifacts with version control
import datetime

def save_training_run(model, config, metrics, output_dir: str = 'models') -> str:
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    run_id = f"{config.model_name}_v{config.version}_{timestamp}"
    os.makedirs(output_dir, exist_ok=True)
    model_path = f"{output_dir}/{run_id}.joblib"
    joblib.dump(model, model_path)
    metadata = {
        'run_id': run_id,
        'config': config.__dict__,
        'metrics': metrics,
        'timestamp': timestamp,
        'model_path': model_path,
    }
    metadata_path = f"{output_dir}/{run_id}_metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    logger.info(f"Training run saved: {run_id}")
    return run_id

run_id = save_training_run(model, config, metrics)
print(f"\nTraining run ID: {run_id}")
print("Files saved to models/ directory")

## 4. Building a Simple Prediction Pipeline

A prediction pipeline encapsulates all preprocessing and inference logic into a clean, reusable interface. This is what you'll actually deploy to production.

### Design Principles:
- **Single responsibility**: One class/function for predictions
- **Input validation**: Check data before inference
- **Error handling**: Graceful failures with informative messages
- **Batch support**: Handle single samples AND batches efficiently


In [ ]:
class PredictionPipeline:
    def __init__(self, model_path: str):
        artifacts = joblib.load(model_path)
        self.model = artifacts['model']
        self.scaler = artifacts['scaler']
        self.feature_names = artifacts['feature_names']
        self.metadata = artifacts.get('training_metadata', {})

    def validate_input(self, X: np.ndarray) -> np.ndarray:
        X = np.array(X)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        if X.shape[1] != len(self.feature_names):
            raise ValueError(
                f"Expected {len(self.feature_names)} features, got {X.shape[1]}. "
                f"Features: {self.feature_names}"
            )
        return X

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = self.validate_input(X)
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        X = self.validate_input(X)
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)

    def predict_batch(self, X_list: list) -> Dict[str, Any]:
        X = np.array(X_list)
        predictions = self.predict(X)
        probabilities = self.predict_proba(X)
        return {
            'predictions': predictions.tolist(),
            'probabilities': probabilities.tolist(),
            'model_version': self.metadata.get('model_version'),
            'inference_samples': len(X_list),
        }

    def get_model_info(self) -> Dict[str, Any]:
        return self.metadata

pipeline = PredictionPipeline('models/complete_pipeline_v1.joblib')
print("Pipeline ready for production inference")

In [ ]:
# Test the prediction pipeline
sample_single = X_test[0]
pred_single = pipeline.predict(sample_single)
print(f"Single prediction: {pred_single[0]}")

sample_batch = X_test[:5]
batch_result = pipeline.predict_batch(sample_batch.tolist())
print("\nBatch prediction result:")
print(f" Predictions: {batch_result['predictions']}")
print(f" Model version: {batch_result['model_version']}")
print(f" Samples processed: {batch_result['inference_samples']}")

proba = pipeline.predict_proba(sample_single)
print(f"\nPrediction probabilities: {proba[0]}")

## 5. Basic Model Deployment Options

Now that we have a serialized model and a prediction pipeline, let's explore how to serve it. We'll cover three approaches:
1. **Direct loading** (simplest, for batch jobs)
2. **Streamlit** (interactive web apps)
3. **Flask API** (backend service integration)


### 5.1 Saving and Loading Models (Recap)

The foundation of all deployment strategies is reliable serialization. We've covered joblib for scikit-learn models. Remember:
- Always save preprocessing steps WITH the model
- Include metadata (version, training date, metrics)
- Use deterministic filenames with timestamps


In [ ]:
# Production inference function (standalone script style)
def load_model_for_inference(model_path: str):
    try:
        artifacts = joblib.load(model_path)
        logger.info(f"Model loaded successfully from {model_path}")
        return artifacts
    except FileNotFoundError:
        logger.error(f"Model file not found: {model_path}")
        raise
    except Exception as e:
        logger.error(f"Error loading model: {str(e)}")
        raise

def run_inference(model_artifacts: dict, input_data: np.ndarray) -> np.ndarray:
    model = model_artifacts['model']
    scaler = model_artifacts['scaler']
    if input_data.ndim == 1:
        input_data = input_data.reshape(1, -1)
    input_scaled = scaler.transform(input_data)
    return model.predict(input_scaled)

artifacts = load_model_for_inference('models/complete_pipeline_v1.joblib')
new_data = np.random.randn(3, 10)
predictions = run_inference(artifacts, new_data)
print(f"Production inference result: {predictions}")

### 5.2 Creating a Prediction Function / Class

For production, wrap everything in a clean interface. Here's the pattern:


In [ ]:
# Production-ready predictor class
class ProductionPredictor:
    def __init__(self, model_path: str):
        self.pipeline = PredictionPipeline(model_path)
        self.model_info = self.pipeline.get_model_info()

    def predict(self, features: list) -> dict:
        try:
            result = self.pipeline.predict_batch(features)
            result['status'] = 'success'
            result['error'] = None
            return result
        except Exception as e:
            logger.error(f"Prediction error: {str(e)}")
            return {'status': 'error', 'error': str(e), 'predictions': None}

    def health_check(self) -> dict:
        return {
            'status': 'healthy',
            'model_version': self.model_info.get('model_version'),
            'model_type': type(self.pipeline.model).__name__,
        }

predictor = ProductionPredictor('models/complete_pipeline_v1.joblib')
print("Health Check:", predictor.health_check())
test_features = [[0.5, -1.2, 3.4, 0.1, -0.8, 1.2, 0.0, 2.1, -0.5, 1.0]]
result = predictor.predict(test_features)
print(f"\nPrediction Result: {result}")

### 5.3 Simple Streamlit Demo (Code Structure)

Streamlit is perfect for quickly creating interactive ML demos. Save this as `app.py` and run with `streamlit run app.py`.


In [ ]:
# Save Streamlit app code to a file for reference
streamlit_code = '''
import streamlit as st
import numpy as np
from production_predictor import ProductionPredictor

st.set_page_config(page_title="ML Model Demo", page_icon=":robot_face:")
st.title("ML Model Deployment Demo")
st.subheader("Day 66: MLOps Basics")

@st.cache_resource
def load_model():
    return ProductionPredictor('models/complete_pipeline_v1.joblib')

predictor = load_model()
st.sidebar.header("Model Information")
health = predictor.health_check()
st.sidebar.write(f"**Version:** {health['model_version']}")
st.sidebar.write(f"**Type:** {health['model_type']}")
st.sidebar.write(f"**Status:** {health['status']}")

st.header("Enter Feature Values")
cols = st.columns(5)
features = []
for i in range(10):
    with cols[i % 5]:
        val = st.number_input(f"Feature {i}", value=0.0, key=f"feat_{i}")
        features.append(val)

if st.button("Predict", type="primary"):
    with st.spinner("Running inference..."):
        result = predictor.predict([features])
    if result['status'] == 'success':
        st.success(f"Prediction: {result['predictions'][0]}")
        st.write("Probabilities:", result['probabilities'][0])
    else:
        st.error(f"Error: {result['error']}")

st.header("Batch Prediction")
uploaded_file = st.file_uploader("Upload CSV with features", type=['csv'])
if uploaded_file:
    import pandas as pd
    df = pd.read_csv(uploaded_file)
    st.write("Preview:", df.head())
    if st.button("Process Batch"):
        batch_result = predictor.predict(df.values.tolist())
        df['prediction'] = batch_result['predictions']
        st.write("Results:", df)
        st.download_button("Download Results", df.to_csv(index=False), "predictions.csv")
'''

with open('streamlit_app.py', 'w') as f:
    f.write(streamlit_code)

print("Streamlit app code saved to: streamlit_app.py")
print("\nTo run: streamlit run streamlit_app.py")

### 5.4 Flask API Basics (Structure)

For backend integration, a REST API is the standard. Here's the complete Flask structure:


In [ ]:
# Save Flask API code to a file for reference
flask_code = '''
from flask import Flask, request, jsonify
from production_predictor import ProductionPredictor
import logging

app = Flask(__name__)
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

predictor = ProductionPredictor('models/complete_pipeline_v1.joblib')
logger.info("API server initialized")

@app.route('/health', methods=['GET'])
def health_check():
    return jsonify(predictor.health_check())

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        if not data or 'features' not in data:
            return jsonify({'error': 'Missing features field'}), 400
        features = data['features']
        if isinstance(features[0], (int, float)):
            features = [features]
        result = predictor.predict(features)
        return jsonify(result)
    except Exception as e:
        logger.error(f"Prediction error: {str(e)}")
        return jsonify({'status': 'error', 'error': str(e)}), 500

@app.route('/info', methods=['GET'])
def model_info():
    return jsonify({'model_info': predictor.model_info, 'endpoints': ['/health', '/predict', '/info']})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
'''

with open('flask_api.py', 'w') as f:
    f.write(flask_code)

print("Flask API code saved to: flask_api.py")
print("\nTo run: python flask_api.py")
print("\nAPI Endpoints:")
print(" GET /health - Health check")
print(" POST /predict - Make predictions")
print(" GET /info - Model metadata")

## 6. Versioning Models and Data

Version control for ML is more complex than code because you must track:
- **Code version** (git commit)
- **Data version** (dataset hash/timestamp)
- **Model version** (training run ID)
- **Environment** (dependencies, OS)

### Simple Versioning Strategy:
Use a structured naming convention and metadata files.


In [ ]:
import hashlib
import json

class ModelVersionManager:
    def __init__(self, registry_dir: str = 'model_registry'):
        self.registry_dir = registry_dir
        os.makedirs(registry_dir, exist_ok=True)
        self.registry_file = f"{registry_dir}/registry.json"
        self.registry = self._load_registry()

    def _load_registry(self) -> dict:
        if os.path.exists(self.registry_file):
            with open(self.registry_file, 'r') as f:
                return json.load(f)
        return {'models': [], 'active_model': None}

    def _save_registry(self):
        with open(self.registry_file, 'w') as f:
            json.dump(self.registry, f, indent=2)

    def _compute_hash(self, file_path: str) -> str:
        hash_md5 = hashlib.md5()
        with open(file_path, 'rb') as f:
            for chunk in iter(lambda: f.read(4096), b''):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()

    def register_model(self, model_path: str, metrics: dict, tags: list = None) -> str:
        timestamp = datetime.datetime.now().isoformat()
        model_hash = self._compute_hash(model_path)
        version = f"v{len(self.registry['models']) + 1}.0"
        entry = {
            'version': version,
            'path': model_path,
            'hash': model_hash,
            'timestamp': timestamp,
            'metrics': metrics,
            'tags': tags or [],
            'status': 'staging',
        }
        self.registry['models'].append(entry)
        self._save_registry()
        return version

    def promote_model(self, version: str, stage: str = 'production'):
        for model in self.registry['models']:
            if model['version'] == version:
                model['status'] = stage
                if stage == 'production':
                    self.registry['active_model'] = version
                self._save_registry()
                return
        raise ValueError(f"Version {version} not found")

    def get_active_model(self) -> dict:
        active_version = self.registry.get('active_model')
        if not active_version:
            return None
        for model in self.registry['models']:
            if model['version'] == active_version:
                return model
        return None

    def list_models(self) -> list:
        return self.registry['models']

version_manager = ModelVersionManager()
v1 = version_manager.register_model(
    'models/complete_pipeline_v1.joblib',
    metrics={'accuracy': 0.92},
    tags=['baseline', 'random_forest'],
)
print(f"Registered: {v1}")
version_manager.promote_model(v1, 'production')
active = version_manager.get_active_model()
print(f"\nActive model: {active['version']} (status: {active['status']})")
print(f"\nRegistry contains {len(version_manager.list_models())} model(s)")

## 7. Monitoring and Logging Basics

Production ML systems MUST be monitored. Unlike traditional software, ML models degrade silently as data distributions shift.

### What to Monitor:
- **Prediction latency** (response time)
- **Prediction distribution** (are outputs shifting?)
- **Input data distribution** (feature drift)
- **Error rates** (failed predictions)
- **Business metrics** (downstream impact)

### Logging Strategy:
- Structured logs (JSON format)
- Correlation IDs for request tracing
- Separate application logs from access logs


In [ ]:
import time
from collections import defaultdict
import json

class ModelMonitor:
    def __init__(self, model_version: str):
        self.model_version = model_version
        self.prediction_count = 0
        self.error_count = 0
        self.latency_history = []
        self.prediction_distribution = defaultdict(int)
        self.start_time = time.time()

    def log_prediction(self, features: np.ndarray, prediction: int, probability: float = None, latency_ms: float = None):
        self.prediction_count += 1
        self.prediction_distribution[str(prediction)] += 1
        if latency_ms:
            self.latency_history.append(latency_ms)
        log_entry = {
            'timestamp': datetime.datetime.now().isoformat(),
            'model_version': self.model_version,
            'prediction': int(prediction),
            'probability': float(probability) if probability else None,
            'latency_ms': latency_ms,
            'feature_summary': {'mean': float(np.mean(features)), 'std': float(np.std(features))},
        }
        logger.info(f"Prediction logged: {json.dumps(log_entry)}")

    def log_error(self, error_type: str, error_message: str):
        self.error_count += 1
        logger.error(f"[{error_type}] {error_message}")

    def get_metrics(self) -> dict:
        uptime = time.time() - self.start_time
        avg_latency = np.mean(self.latency_history) if self.latency_history else 0
        return {
            'model_version': self.model_version,
            'uptime_seconds': uptime,
            'total_predictions': self.prediction_count,
            'total_errors': self.error_count,
            'error_rate': self.error_count / max(self.prediction_count, 1),
            'avg_latency_ms': avg_latency,
            'prediction_distribution': dict(self.prediction_distribution),
            'requests_per_minute': (self.prediction_count / uptime) * 60 if uptime > 0 else 0,
        }

    def check_health(self) -> dict:
        metrics = self.get_metrics()
        is_healthy = (metrics['error_rate'] < 0.05 and metrics['avg_latency_ms'] < 1000)
        return {'status': 'healthy' if is_healthy else 'degraded', 'metrics': metrics, 'alerts': [] if is_healthy else ['High error rate or latency detected']}

monitor = ModelMonitor(model_version='v1.0')
for i in range(10):
    sample = X_test[i]
    start = time.time()
    pred = pipeline.predict(sample)[0]
    proba = pipeline.predict_proba(sample)[0][int(pred)]
    latency = (time.time() - start) * 1000
    monitor.log_prediction(sample, pred, proba, latency)

metrics = monitor.get_metrics()
print("Monitoring Metrics:")
print(json.dumps(metrics, indent=2))
health = monitor.check_health()
print(f"\nHealth Status: {health['status']}")

## 8. End-to-End Example: Train -> Save -> Load -> Predict Pipeline

Let's put it all together in a complete workflow that demonstrates the full MLOps cycle.


In [ ]:
# ============================================================
# COMPLETE END-TO-END MLOps PIPELINE
# ============================================================

print("Starting End-to-End MLOps Pipeline")
print("=" * 50)

print("\nStep 1: Configuration")
config = ModelConfig(n_estimators=150, max_depth=12, model_name="production_rf", version="2.0.0")
print(f"Config: {config}")

print("\nStep 2: Data Loading")
set_seeds(config.random_state)
X_train, X_test, y_train, y_test = load_and_split_data(config)

print("\nStep 3: Preprocessing")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Scaler fitted and applied")

print("\nStep 4: Model Training")
model = RandomForestClassifier(n_estimators=config.n_estimators, max_depth=config.max_depth, random_state=config.random_state)
model.fit(X_train_scaled, y_train)
print("Model training complete")

print("\nStep 5: Evaluation")
metrics = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test)
print(f"Final metrics: {metrics}")

In [ ]:
# Step 6: Save complete pipeline
print("\nStep 6: Saving Pipeline")
pipeline_artifacts = {
    'model': model,
    'scaler': scaler,
    'feature_names': [f'feature_{i}' for i in range(X_train.shape[1])],
    'training_metadata': {
        'n_samples': len(X_train),
        'n_features': X_train.shape[1],
        'train_score': metrics['train_accuracy'],
        'test_score': metrics['test_accuracy'],
        'model_version': config.version,
        'model_name': config.model_name,
        'training_date': datetime.datetime.now().isoformat(),
        'config': config.__dict__,
    },
}

model_path = f"models/{config.model_name}_v{config.version}.joblib"
joblib.dump(pipeline_artifacts, model_path)
print(f"Saved to: {model_path}")

print("\nStep 7: Version Registration")
version_manager = ModelVersionManager()
version_id = version_manager.register_model(model_path, metrics=metrics, tags=['production_ready', 'e2e_demo'])
version_manager.promote_model(version_id, 'production')
print(f"Model promoted to production: {version_id}")

In [ ]:
# Step 8: Load and serve
print("\nStep 8: Loading for Inference")
inference_pipeline = PredictionPipeline(model_path)
print("Pipeline loaded and ready")

print("\nStep 9: Batch Prediction")
batch_input = X_test[:20].tolist()
batch_result = inference_pipeline.predict_batch(batch_input)
print(f"Processed {batch_result['inference_samples']} predictions")
print(f"Predictions: {batch_result['predictions'][:5]}...")

print("\nStep 10: Monitoring Setup")
monitor = ModelMonitor(model_version=config.version)
for i in range(5):
    sample = X_test[i]
    pred = inference_pipeline.predict(sample)[0]
    monitor.log_prediction(sample, pred)

health = monitor.check_health()
print(f"System health: {health['status']}")

print("\n" + "=" * 50)
print("END-TO-END PIPELINE COMPLETE!")
print("Train -> Save -> Version -> Load -> Predict -> Monitor")
print("=" * 50)

## 9. Best Practices for Moving from Notebook to Production

### DO:
- **Modularize code** into functions and classes (no more 50-cell notebooks!)
- **Use configuration files** (YAML/JSON) instead of hardcoded parameters
- **Version everything**: data, models, code, and environments
- **Write unit tests** for data validation, preprocessing, and inference
- **Containerize** with Docker for environment consistency
- **Implement health checks** and graceful degradation
- **Log structured data** for observability
- **Use feature stores** for consistent preprocessing across train/serve
- **Implement A/B testing** framework for model comparison
- **Plan for rollback** (keep previous model versions hot)

### DON'T:
- Don't deploy notebooks directly (convert to scripts)
- Don't ignore data drift (monitor input distributions)
- Don't skip input validation (garbage in, garbage out)
- Don't hardcode paths (use environment variables)
- Don't forget about security (validate inputs, use HTTPS)
- Don't neglect latency requirements (profile your inference)
- Don't deploy without a rollback plan

### Production Checklist:
1. Model serialized with preprocessing
2. Input validation implemented
3. Error handling and logging added
4. Health check endpoint available
5. Model version registered
6. Monitoring dashboard configured
7. Rollback procedure documented
8. Load testing completed
9. Security review passed
10. Documentation updated

## Hands-On Exercises

Complete these exercises to solidify your MLOps skills. Each exercise builds on the previous one, creating a production-ready system by the end.

### Exercise 1: Save and Load a Trained Model
Train a `GradientBoostingClassifier` on the sample data, save it using joblib, then load it back and verify predictions match.


### Exercise 2: Build a Reusable Prediction Class
Create a `GradientBoostingPredictor` class that encapsulates model loading, input validation, and prediction (similar to our `PredictionPipeline`).


### Exercise 3: Create a Simple Inference Script
Write a standalone Python script (not notebook cell) that loads a saved model and makes predictions on a CSV file. Include argument parsing for the input file path.


### Exercise 4: Add Logging to the Pipeline
Enhance your prediction class from Exercise 2 with comprehensive logging. Log every prediction with timestamp, input shape, and output value.


### Exercise 5: Prepare a Model for Deployment with Preprocessing
Create a complete pipeline that includes both a `StandardScaler` and a `PCA` (n_components=5) step, save the entire pipeline, and demonstrate that loaded pipeline produces identical results.


### Exercise 6: Simulate a Batch Prediction Service
Write a function that accepts a list of 100 samples, processes them in mini-batches of 10, and returns aggregated results with processing time statistics.


### Exercise 7: Design a Basic Model Versioning Strategy
Implement a simple versioning system that tracks model file hashes, training dates, and performance metrics. Include functions to list all versions and retrieve the best-performing model.


### Exercise 8: Create a Model Health Check Function
Design a function that evaluates model "health" based on recent prediction latency, error rate, and output distribution stability. Return a status: 'healthy', 'degraded', or 'critical'.


### Exercise 9: Build a Simple Flask API Skeleton
Create a minimal Flask application with `/predict` and `/health` endpoints. The `/predict` endpoint should accept JSON input, validate it, and return predictions using your model from Exercise 1.


### Exercise 10: Document a Deployment Runbook
Write a markdown cell (or separate file) documenting the step-by-step procedure for deploying your model to production, including rollback instructions and monitoring setup.


## Solutions & Best Practices (Review After Attempting)

Below are detailed solutions and best practices for each exercise. Study these after attempting the exercises yourself.


### Solution 1: Save and Load a Trained Model

**Key Points:**
- Always use `joblib` for scikit-learn models (handles numpy arrays better than pickle)
- Verify by comparing predictions on identical input, not just file existence
- Include metadata when saving


In [ ]:
# Solution 1
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(n_estimators=50, random_state=42)
gb_model.fit(X_train, y_train)
print(f"Train acc: {gb_model.score(X_train, y_train):.4f}")
print(f"Test acc: {gb_model.score(X_test, y_test):.4f}")

joblib.dump(gb_model, 'models/gb_model_v1.joblib')
print("Model saved")

loaded_gb = joblib.load('models/gb_model_v1.joblib')
original_preds = gb_model.predict(X_test[:10])
loaded_preds = loaded_gb.predict(X_test[:10])
print(f"Predictions match: {np.array_equal(original_preds, loaded_preds)}")

### Solution 2: Build a Reusable Prediction Class

**Best Practices:**
- Validate input dimensions against training features
- Handle both single samples and batches
- Include informative error messages
- Add a `predict_proba` method for probability outputs


In [ ]:
# Solution 2
class GradientBoostingPredictor:
    def __init__(self, model_path: str):
        self.model = joblib.load(model_path)
        self.expected_features = 10

    def validate(self, X):
        X = np.array(X)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        if X.shape[1] != self.expected_features:
            raise ValueError(f"Expected {self.expected_features} features, got {X.shape[1]}")
        return X

    def predict(self, X):
        return self.model.predict(self.validate(X))

    def predict_proba(self, X):
        return self.model.predict_proba(self.validate(X))

gb_predictor = GradientBoostingPredictor('models/gb_model_v1.joblib')
print(gb_predictor.predict(X_test[0]))
print(gb_predictor.predict_proba(X_test[:3]))

### Solution 3: Create a Simple Inference Script

**Best Practices:**
- Use `argparse` for CLI interfaces
- Validate file existence before loading
- Output results to stdout or file
- Exit with appropriate error codes


In [ ]:
# Solution 3 (save as inference_script.py)
inference_script = '''
import argparse
import pandas as pd
import numpy as np
import joblib


def main():
    parser = argparse.ArgumentParser(description='Run model inference')
    parser.add_argument('--model', required=True, help='Path to model file')
    parser.add_argument('--input', required=True, help='Path to input CSV')
    parser.add_argument('--output', default='predictions.csv', help='Output file')
    args = parser.parse_args()

    model = joblib.load(args.model)
    df = pd.read_csv(args.input)
    predictions = model.predict(df.values)

    df['prediction'] = predictions
    df.to_csv(args.output, index=False)
    print(f"Predictions saved to {args.output}")


if __name__ == '__main__':
    main()
'''

with open('inference_script.py', 'w') as f:
    f.write(inference_script)

print("Inference script saved to: inference_script.py")
print("Usage: python inference_script.py --model models/gb_model_v1.joblib --input data.csv")

### Solution 4: Add Logging to the Pipeline

**Best Practices:**
- Use Python's `logging` module, not print statements
- Log at appropriate levels (INFO for predictions, ERROR for failures)
- Include structured data (JSON) for log aggregation
- Don't log sensitive input data in production


In [ ]:
# Solution 4
import logging
import json

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler('model_predictions.log'), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

class LoggedPredictor(GradientBoostingPredictor):
    def predict(self, X):
        X = self.validate(X)
        logger.info(f"Input shape: {X.shape}")
        try:
            preds = self.model.predict(X)
            logger.info(f"Predictions: {preds}")
            return preds
        except Exception as e:
            logger.error(f"Prediction failed: {str(e)}")
            raise

logged_pred = LoggedPredictor('models/gb_model_v1.joblib')
result = logged_pred.predict(X_test[:3])
print("Check model_predictions.log for entries")

### Solution 5: Prepare a Model for Deployment with Preprocessing

**Best Practices:**
- Always fit preprocessing on training data only
- Save the ENTIRE pipeline (preprocessing + model)
- Verify that loaded pipeline handles new data identically
- Document expected input format


In [ ]:
# Solution 5
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

pca_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=5)),
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42)),
])

pca_pipeline.fit(X_train, y_train)
print(f"Pipeline test accuracy: {pca_pipeline.score(X_test, y_test):.4f}")

joblib.dump(pca_pipeline, 'models/pca_pipeline_v1.joblib')
loaded_pipeline = joblib.load('models/pca_pipeline_v1.joblib')
original_score = pca_pipeline.score(X_test, y_test)
loaded_score = loaded_pipeline.score(X_test, y_test)
print(f"Scores match: {abs(original_score - loaded_score) < 1e-10}")
print(f"Loaded pipeline prediction: {loaded_pipeline.predict(X_test[:1])}")

### Solution 6: Simulate a Batch Prediction Service

**Best Practices:**
- Process in chunks to manage memory
- Track timing for performance monitoring
- Handle partial failures (don't fail entire batch)
- Return metadata with results


In [ ]:
# Solution 6
def batch_predict_service(model, data, batch_size=10):
    results, timings, errors = [], [], []
    for i in range(0, len(data), batch_size):
        batch = data[i:i + batch_size]
        start = time.time()
        try:
            preds = model.predict(batch)
            results.extend(preds.tolist())
            timings.append((time.time() - start) * 1000)
        except Exception as e:
            errors.append({'batch': i // batch_size, 'error': str(e)})
            results.extend([None] * len(batch))
    return {
        'predictions': results,
        'total_samples': len(data),
        'batches': len(timings),
        'avg_latency_ms': np.mean(timings),
        'max_latency_ms': max(timings) if timings else 0,
        'errors': errors,
    }

test_100 = X_test[:100]
service_result = batch_predict_service(loaded_pipeline, test_100, batch_size=10)
print(f"Processed {service_result['total_samples']} samples in {service_result['batches']} batches")
print(f"Average latency: {service_result['avg_latency_ms']:.2f}ms")
print(f"Errors: {len(service_result['errors'])}")

### Solution 7: Design a Basic Model Versioning Strategy

**Best Practices:**
- Use semantic versioning (MAJOR.MINOR.PATCH)
- Store hashes for integrity verification
- Track performance metrics for each version
- Maintain a registry file (JSON/YAML)
- Support model promotion (staging -> production)


In [ ]:
# Solution 7
class SimpleModelRegistry:
    def __init__(self, path='model_registry.json'):
        self.path = path
        self.registry = self._load()

    def _load(self):
        if os.path.exists(self.path):
            with open(self.path) as f:
                return json.load(f)
        return {'models': {}}

    def _save(self):
        with open(self.path, 'w') as f:
            json.dump(self.registry, f, indent=2)

    def register(self, name, version, path, metrics):
        if name not in self.registry['models']:
            self.registry['models'][name] = {}
        self.registry['models'][name][version] = {
            'path': path,
            'metrics': metrics,
            'registered_at': datetime.datetime.now().isoformat(),
        }
        self._save()
        return version

    def get_best(self, name, metric='accuracy'):
        versions = self.registry['models'].get(name, {})
        if not versions:
            return None
        return max(versions.items(), key=lambda x: x[1]['metrics'].get(metric, 0))

    def list_versions(self, name):
        return list(self.registry['models'].get(name, {}).keys())

registry = SimpleModelRegistry()
registry.register('rf_model', 'v1.0', 'models/rf_v1.joblib', {'accuracy': 0.89})
registry.register('rf_model', 'v2.0', 'models/rf_v2.joblib', {'accuracy': 0.92})
best = registry.get_best('rf_model')
print(f"Best model: {best[0]} with accuracy {best[1]['metrics']['accuracy']}")

### Solution 8: Create a Model Health Check Function

**Best Practices:**
- Define thresholds based on SLAs
- Monitor both technical and business metrics
- Alert on anomalies, not just hard failures
- Include historical comparison (is this abnormal?)


In [ ]:
# Solution 8
def model_health_check(recent_latencies, error_count, total_requests, output_distribution, latency_threshold=500, error_threshold=0.05):
    avg_latency = np.mean(recent_latencies) if recent_latencies else 0
    latency_status = avg_latency < latency_threshold
    error_rate = error_count / max(total_requests, 1)
    error_status = error_rate < error_threshold
    dist_values = list(output_distribution.values())
    dist_balance = min(dist_values) / max(dist_values) if max(dist_values) > 0 else 0
    dist_status = dist_balance > 0.1
    checks = [latency_status, error_status, dist_status]
    if all(checks):
        return 'healthy'
    elif sum(checks) >= 2:
        return 'degraded'
    return 'critical'

status = model_health_check(
    recent_latencies=[120, 150, 130, 200, 180],
    error_count=2,
    total_requests=100,
    output_distribution={0: 45, 1: 55},
)
print(f"Health status: {status}")

### Solution 9: Build a Simple Flask API Skeleton

**Best Practices:**
- Validate all inputs (type, shape, range)
- Return consistent JSON structures
- Use HTTP status codes correctly
- Never expose stack traces to clients
- Add request ID tracking for debugging


In [ ]:
# Solution 9
flask_solution = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np
import uuid

app = Flask(__name__)
model = joblib.load('models/gb_model_v1.joblib')

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy', 'model_loaded': True})

@app.route('/predict', methods=['POST'])
def predict():
    request_id = str(uuid.uuid4())
    try:
        data = request.get_json()
        if 'features' not in data:
            return jsonify({'error': 'Missing features', 'request_id': request_id}), 400
        features = np.array(data['features'])
        if features.ndim == 1:
            features = features.reshape(1, -1)
        predictions = model.predict(features)
        probabilities = model.predict_proba(features)
        return jsonify({
            'request_id': request_id,
            'predictions': predictions.tolist(),
            'probabilities': probabilities.tolist(),
            'status': 'success',
        })
    except Exception as e:
        return jsonify({'request_id': request_id, 'error': str(e), 'status': 'error'}), 500

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
'''

with open('flask_solution.py', 'w') as f:
    f.write(flask_solution)

print("Flask solution saved to: flask_solution.py")

### Solution 10: Document a Deployment Runbook

**Best Practices:**
- Include pre-deployment checklist
- Document rollback steps explicitly
- Define monitoring and alerting rules
- Specify who to contact for issues
- Include post-deployment verification steps


In [ ]:
# Solution 10
runbook = """
# Model Deployment Runbook: RandomForest v2.0

## Pre-Deployment Checklist
- [ ] Model tested on holdout set (accuracy > 0.90)
- [ ] Preprocessing pipeline included in artifact
- [ ] Unit tests pass (test_inference.py)
- [ ] Model file hash verified against registry
- [ ] Rollback model (v1.0) verified and ready

## Deployment Steps
1. Copy model artifact to production storage: `s3://models/production/`
2. Update environment variable: `MODEL_PATH=/models/rf_v2.joblib`
3. Restart prediction service: `sudo systemctl restart prediction-api`
4. Verify health endpoint: `curl http://localhost:5000/health`
5. Run smoke tests: `python tests/smoke_test.py`

## Monitoring (First 24 Hours)
- Watch error rate (alert if > 1%)
- Monitor p95 latency (alert if > 200ms)
- Check prediction distribution hourly
- Review logs for anomalies

## Rollback Procedure
1. Set `MODEL_PATH=/models/rf_v1.joblib` (previous version)
2. Restart service
3. Verify health endpoint
4. Alert team via Slack #ml-alerts
5. Create incident ticket for investigation

## Post-Deployment
- [ ] Update model registry (mark v2.0 as production)
- [ ] Notify stakeholders
- [ ] Schedule 1-week performance review
"""

with open('deployment_runbook.md', 'w') as f:
    f.write(runbook)

print("Deployment runbook saved to: deployment_runbook.md")

---

## Day 66 Complete!

You have now taken your first major step from modeling to production - a critical transition in real AI engineering. Today you learned:

- How to serialize and version models reliably
- Build reproducible training pipelines
- Create production-ready prediction interfaces
- Use deployment options from batch scripts to APIs
- Apply basic monitoring and health checks
- Understand the complete MLOps lifecycle from training to serving

### Tomorrow's Teaser
**Day 67: Advanced Ensembling & Stacking Techniques** - Learn how to combine multiple models using voting, stacking, and blending strategies.

---

**Python & AI Learning Path | Day 66 / 369**

*Keep building. Keep deploying. Keep learning.*